# Lección 7.2 — Early stopping: saber cuándo parar

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.4 (épocas e iteraciones), Lección 3.7 (curvas de aprendizaje), Lección 6.3 (boosting), Lección 7.1 (regularización)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué es **early stopping** (parada temprana) y por qué es una forma de regularización.
- Leer una curva de entrenamiento/validación y señalar **dónde** conviene parar.
- Usar y ajustar sus perillas: **paciencia** (`n_iter_no_change`), `validation_fraction` y `tol`.
- Activarlo en tres modelos de scikit-learn: `HistGradientBoosting`, `SGDClassifier` y `MLPClassifier`.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, make_friedman1
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

warnings.filterwarnings("ignore")          # ocultamos avisos de convergencia para no distraernos
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: las galletas en el horno

Estás horneando galletas. La receta dice "entre 10 y 15 minutos", pero cada horno es distinto. ¿Qué haces?

- **No** pones un temporizador de 40 minutos y te vas: saldrían quemadas.
- **Miras por la ventanita** cada minuto. Mientras se doren más, las dejas. En cuanto dejan de mejorar (o empiezan a oscurecerse), **las sacas**.

Entrenar un modelo **por pasos** (iteraciones, épocas, árboles que se suman uno a uno) es igual:
- Al principio, cada paso **mejora** el modelo: aprende el patrón.
- Después de cierto punto, los pasos extra solo sirven para **memorizar el ruido**: el modelo "se quema" (sobreajuste, Lección 4.4).

**Early stopping** (*parada temprana*) es mirar por la ventanita: reservas un pedacito de datos de **validación**, lo revisas después de cada paso y **detienes** el entrenamiento cuando deja de mejorar.

### ✋ Ejercicio 1 — Calentamiento
En la analogía, ¿qué representan…
1. …la **ventanita** del horno?
2. …las galletas **quemadas**?
3. …el **temporizador de 40 minutos**?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El conjunto de **validación**: una forma de ver cómo va el modelo con datos que **no** usa para aprender.
2. Un modelo **sobreajustado**: siguió entrenando hasta memorizar el ruido.
3. Un número fijo de iteraciones (`max_iter`, `n_estimators`, épocas) elegido sin mirar los datos: puede quedarse corto o pasarse.
</details>

---
## 2. 📖 El concepto: la curva de validación tiene forma de "U"

Vamos a verlo en un modelo que se entrena por pasos: **gradient boosting** (Lección 6.3), que suma árboles uno a uno. Usaremos `HistGradientBoostingClassifier`, la versión rápida de scikit-learn.

Datos: 3,000 clientes ficticios con 20 características, de las que solo 6 importan, y un **15 % de etiquetas al azar** (ruido). Separamos:
- **entrenamiento** → para aprender,
- **validación** → la ventanita,
- **prueba** → el examen final, guardado bajo llave.

In [ ]:
X, y = make_classification(n_samples=3_000, n_features=20, n_informative=6,
                           flip_y=0.15, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)       # prueba: bajo llave
X_ap, X_val, y_ap, y_val = train_test_split(X_tr, y_tr, test_size=0.25, random_state=0)  # aprender / validar
print(f"Aprender: {len(X_ap)} · Validación: {len(X_val)} · Prueba: {len(X_te)}")

### 🔮 Ejercicio 2 — Dibuja la curva antes de verla
Vamos a entrenar **500 árboles** sin early stopping y medir la **pérdida** (el error, Lección 3.2: más bajo = mejor) después de cada árbol, en los datos de aprender y en los de validación.

Antes de ejecutar, dibuja en papel cómo crees que serán las dos curvas.

✍️ *Mi dibujo / mi descripción:*

In [ ]:
modelo = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.1,
                                        early_stopping=False, random_state=0).fit(X_ap, y_ap)

# staged_predict_proba nos da las predicciones "como si" hubiéramos parado en el árbol 1, 2, 3...
perdida_ap  = [log_loss(y_ap,  p) for p in modelo.staged_predict_proba(X_ap)]
perdida_val = [log_loss(y_val, p) for p in modelo.staged_predict_proba(X_val)]
mejor = int(np.argmin(perdida_val)) + 1

plt.plot(range(1, 501), perdida_ap, label="Aprender (entrenamiento)")
plt.plot(range(1, 501), perdida_val, label="Validación")
plt.axvline(mejor, color="k", ls="--", label=f"mejor punto: árbol {mejor}")
plt.xlabel("Número de árboles (iteraciones)"); plt.ylabel("Pérdida (más bajo = mejor)")
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Pérdida de validación en el mejor punto ({mejor} árboles): {min(perdida_val):.3f}")
print(f"Pérdida de validación con los 500 árboles           : {perdida_val[-1]:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- La curva de **entrenamiento** baja y baja casi hasta cero: con suficientes árboles, el modelo memoriza todo.
- La de **validación** baja al principio, toca fondo alrededor del **árbol 43** (≈ 0.375) y después **sube** sin parar hasta ≈ 0.84 con 500 árboles: más del doble de error.

Todo lo que el modelo "aprendió" después del árbol 43 fue **ruido**. Lo ideal habría sido sacar las galletas ahí.
</details>

### ✋ Ejercicio 3 — Lee la curva
1. Si solo miraras la curva de entrenamiento, ¿cuántos árboles elegirías? ¿Por qué sería mala idea?
2. Early stopping es una forma de **regularización** (Lección 7.1). ¿Qué "complejidad" está limitando?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Elegirías **todos** (500 o más): la pérdida de entrenamiento nunca deja de bajar. Sería mala idea porque el entrenamiento no ve el sobreajuste; para eso existe la validación.
2. Limita **cuánto tiempo** tiene el modelo para acomodarse a los datos. En boosting, cada árbol extra añade complejidad; en redes neuronales, cada época extra deja que los pesos crezcan y se especialicen. Parar antes mantiene el modelo más simple, igual que cobrar por los coeficientes en Ridge.
</details>

---
## 3. 📖 Cómo funciona: paciencia, tolerancia y validación

En la práctica no entrenas 500 pasos y luego miras la gráfica: el modelo **se vigila solo**. El algoritmo es:

1. Aparta automáticamente un `validation_fraction` de los datos de entrenamiento (p. ej. 10 %).
2. Después de cada paso, mide el desempeño en esa validación.
3. Si mejora **más que** `tol` (una tolerancia mínima), anota "¡nuevo récord!".
4. Si pasan `n_iter_no_change` pasos seguidos **sin récord**, se detiene. A esto se le llama **paciencia** (*patience*).

```
validación:  0.65 0.52 0.44 0.40 0.39 0.39 0.40 0.39 0.41 0.42 0.43 ✋ PARA
             └──────── mejora ────────┘ └─── 5 pasos sin récord (paciencia = 5) ───┘
```

¿Por qué esperar en lugar de parar al primer paso malo? Porque la curva de validación **tiembla**: un paso peor puede ser mala suerte, no sobreajuste. La paciencia evita confundir un bache con una subida.

En `HistGradientBoostingClassifier` basta con poner `early_stopping=True`. El atributo `n_iter_` te dice cuántos pasos dio en realidad.

### 🔮 Ejercicio 4 — ¿Cuántos árboles usará?
Ahora entrenamos con `max_iter=500` pero **con early stopping** (paciencia 10, que es el valor por defecto). Esta vez le pasamos todo el entrenamiento: él solo aparta su validación.

1. ¿Cuántos árboles crees que usará: cerca de 10, cerca de 50 o cerca de 500?
2. ¿Su exactitud y su pérdida en prueba serán mejores o peores que las del modelo con 500 árboles?

✍️ *Mis predicciones:*

In [ ]:
inicio = time.time()
sin_es = HistGradientBoostingClassifier(max_iter=500, early_stopping=False, random_state=0).fit(X_tr, y_tr)
t_sin = time.time() - inicio

inicio = time.time()
con_es = HistGradientBoostingClassifier(max_iter=500, early_stopping=True, n_iter_no_change=10,
                                        validation_fraction=0.1, random_state=0).fit(X_tr, y_tr)
t_con = time.time() - inicio

for nombre, m, t in [("Sin early stopping", sin_es, t_sin), ("Con early stopping", con_es, t_con)]:
    print(f"{nombre}: {m.n_iter_:>3} árboles · exactitud prueba {m.score(X_te, y_te):.1%} · "
          f"pérdida prueba {log_loss(y_te, m.predict_proba(X_te)):.3f} · {t:.2f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Usa **55** árboles de 500 posibles: cerca de 50. La validación dejó de mejorar alrededor del árbol 45 y, tras 10 árboles más sin récord, paró.
2. La **exactitud** es prácticamente la misma (≈ 84.7 % contra ≈ 85.3 %), pero la **pérdida** del modelo de 500 árboles es más del doble (≈ 0.86 contra ≈ 0.40). ¿Cómo puede ser? El modelo sobreajustado sigue acertando la clase casi igual de veces, pero sus probabilidades se volvieron **exageradas**: dice "99 % seguro" incluso cuando se equivoca. Y el modelo vigilado tardó varias veces menos en entrenarse.

Fíjate en que no tuviste que **adivinar** el número de árboles: pusiste un `max_iter` generoso y early stopping encontró el punto. Esta es una receta que usarás mucho (Lección 7.4): **`max_iter` alto + early stopping**, en vez de buscar `max_iter` a mano.

> 📝 Pistas finas: el modelo se queda con **todos** los árboles hasta el momento de parar (los 10 de "paciencia" incluidos), no solo hasta el mejor. Con paciencias moderadas casi no importa. En redes neuronales con Keras (Módulo 9) existe la opción `restore_best_weights=True` para volver al mejor punto.
</details>

### ✏️ Ejercicio 5 — Juega con la paciencia
Cambia `PACIENCIAS` y observa cuántos árboles usa el modelo y qué exactitud logra. ¿La paciencia cambia mucho el resultado en este caso?

In [ ]:
PACIENCIAS = [2, 10, 50]      # ✏️ prueba otros valores, p. ej. [1, 5, 100]

for p in PACIENCIAS:
    m = HistGradientBoostingClassifier(max_iter=500, early_stopping=True, n_iter_no_change=p,
                                       random_state=0).fit(X_tr, y_tr)
    mejor_iter = int(np.argmax(m.validation_score_))      # dónde estuvo el récord de validación
    print(f"paciencia={p:>3}: paró en el árbol {m.n_iter_:>3} (récord en el {mejor_iter:>3}) · "
          f"exactitud prueba {m.score(X_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Paciencia 2 → para en el árbol **23**; paciencia 10 → **55**; paciencia 50 → **95**.
- La exactitud en prueba casi no cambia (≈ 85 % en los tres casos).

En boosting, la curva de validación es bastante suave y la subida es lenta, así que la paciencia no es crítica. **Más paciencia = más tiempo de cómputo**, pero menos riesgo de parar por un bache. Ahora verás un modelo donde la paciencia sí importa mucho.
</details>

---
## 4. 📖 ¿Qué pasa si la curva es muy nerviosa? `SGDClassifier`

`SGDClassifier` es un modelo lineal (como la regresión logística o una SVM lineal) que aprende con **descenso de gradiente estocástico** (Lección 3.5): ajusta sus perillas mirando un ejemplo a la vez. Cada **época** recorre todos los datos una vez.

Como aprende "a saltos", su curva de validación **tiembla mucho**. Eso pone a prueba a la paciencia.

### 🔮 Ejercicio 6 — Paciencia corta en una curva nerviosa
Vamos a entrenar `SGDClassifier` con early stopping y la paciencia por defecto (**5**), y luego con paciencia **20**. ¿Cuál crees que tendrá mejor exactitud en prueba? ¿Por cuánto?

✍️ *Mi predicción:*

In [ ]:
for paciencia in [5, 20]:     # ✏️
    sgd = SGDClassifier(early_stopping=True, n_iter_no_change=paciencia, random_state=0).fit(X_tr, y_tr)
    print(f"paciencia={paciencia:>2}: paró en la época {sgd.n_iter_:>2} · exactitud prueba {sgd.score(X_te, y_te):.1%}")

sgd_largo = SGDClassifier(max_iter=1000, tol=None, random_state=0).fit(X_tr, y_tr)   # tol=None: nunca para antes
print(f"sin parar (1000 épocas): exactitud prueba {sgd_largo.score(X_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Con paciencia **5**, paró en la época **7** con apenas ≈ **60 %**: un bache de la curva lo engañó y paró cuando el modelo **todavía estaba aprendiendo**. ¡Subajuste por impaciencia!
- Con paciencia **20**, llegó a la época 32 y ≈ **73 %**.
- Sin parar (1,000 épocas): ≈ **78 %**. Un modelo lineal tiene poca capacidad para sobreajustar, así que aquí entrenar más no hace daño.

Lecciones:
1. **Paciencia demasiado baja + curva nerviosa = parar antes de tiempo.**
2. Early stopping brilla en modelos **flexibles** que sí pueden sobreajustar (boosting, redes). En un modelo lineal su principal ventaja es ahorrar tiempo con datos enormes.
</details>

---
## 5. 📖 Early stopping en redes neuronales: `MLPClassifier`

En las redes neuronales (Módulo 9) early stopping es **la** técnica estándar: casi nadie entrena una red sin vigilarla. Hagamos un adelanto con `MLPClassifier`, la red neuronal sencilla de scikit-learn.

Usamos las dos lunas con ruido (Lección 5.4), pocos datos (600 puntos) y una red **muy grande** para ellos (dos capas de 200 neuronas): la receta perfecta para sobreajustar.

In [ ]:
X_l, y_l = make_moons(n_samples=600, noise=0.35, random_state=0)
X_l_tr, X_l_te, y_l_tr, y_l_te = train_test_split(X_l, y_l, test_size=0.3, random_state=0)
X_l_ap, X_l_val, y_l_ap, y_l_val = train_test_split(X_l_tr, y_l_tr, test_size=0.2, random_state=0)

red = MLPClassifier(hidden_layer_sizes=(200, 200), learning_rate_init=0.01, random_state=0)
hist_ap, hist_val = [], []
for epoca in range(300):                       # entrenamos época por época para dibujar la curva
    red.partial_fit(X_l_ap, y_l_ap, classes=[0, 1])
    hist_ap.append(log_loss(y_l_ap, red.predict_proba(X_l_ap)))
    hist_val.append(log_loss(y_l_val, red.predict_proba(X_l_val)))

plt.plot(hist_ap, label="Aprender"); plt.plot(hist_val, label="Validación")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.legend(); plt.grid(alpha=.3)
plt.title("Red neuronal grande con pocos datos"); plt.show()

### ✋ Ejercicio 7 — ¿Dónde sacarías las galletas?
1. Mirando la gráfica, ¿en qué época aproximadamente pararías?
2. ¿Qué tan grave es seguir hasta la época 300?

✍️ *Mis respuestas:*

In [ ]:
mejor_epoca = int(np.argmin(hist_val)) + 1
print(f"Mejor época según la validación: {mejor_epoca}  (pérdida {min(hist_val):.3f})")
print(f"Pérdida de validación en la época 300: {hist_val[-1]:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Muy pronto: el mínimo de la validación está en la **época 10** (≈ 0.36).
2. Muy grave: en la época 300 la pérdida de validación es ≈ **1.09**, tres veces peor, mientras la de entrenamiento sigue bajando. La red se volvió **demasiado segura de sí misma**: memoriza puntos ruidosos y, cuando se equivoca, se equivoca con muchísima confianza (y la pérdida castiga eso con dureza).
</details>

Ahora lo hacemos como en la vida real: `early_stopping=True` y la red se vigila sola. En `MLPClassifier` la validación usa la **exactitud** (no la pérdida), y guarda su historia en `validation_scores_`.

In [ ]:
red_es = MLPClassifier(hidden_layer_sizes=(200, 200), learning_rate_init=0.01, max_iter=300,
                       early_stopping=True, validation_fraction=0.2, n_iter_no_change=10,
                       random_state=0).fit(X_l_tr, y_l_tr)

print(f"La red con early stopping paró en la época {red_es.n_iter_}")
print(f"Pérdida en PRUEBA · red de 300 épocas: {log_loss(y_l_te, red.predict_proba(X_l_te)):.3f}")
print(f"Pérdida en PRUEBA · red con early stopping: {log_loss(y_l_te, red_es.predict_proba(X_l_te)):.3f}")

plt.plot(range(1, red_es.n_iter_ + 1), red_es.validation_scores_, "o-")
plt.xlabel("Época"); plt.ylabel("Exactitud en validación"); plt.title("Lo que vigiló early stopping")
plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La red paró sola en la época **15** y su pérdida en prueba es mucho mejor (≈ 0.46 contra ≈ 1.21 de la red de 300 épocas). No tuviste que mirar ninguna gráfica ni adivinar el número de épocas.

> 📝 Si miras la exactitud en prueba en lugar de la pérdida, la diferencia es menor: la red de 300 épocas acierta casi lo mismo, pero sus probabilidades son exageradas ("100 % seguro" cuando se equivoca). Si tu negocio usa esas probabilidades (por ejemplo, para fijar un umbral, Lección 4.3), esa sobreconfianza es un problema serio.
</details>

---
## 6. 📖 Hiperparámetros y perillas

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `early_stopping` | Activa la vigilancia | — | — | `True` en boosting y redes. ⚠️ En `HistGradientBoosting` el valor `"auto"` solo lo activa con **más de 10,000 filas** |
| `n_iter_no_change` (paciencia) | Pasos sin mejora antes de parar | Más tiempo, menos riesgo de parar por un bache | Más rápido, riesgo de parar **antes de tiempo** | 10 (boosting, redes); 10–20 en SGD |
| `validation_fraction` | Parte del entrenamiento apartada para vigilar | Vigilancia más fiable, pero menos datos para aprender | Más datos para aprender, vigilancia más ruidosa | 0.1 – 0.2 |
| `tol` | Mejora mínima que cuenta como "récord" | Para antes (exige mejoras grandes) | Para después | El valor por defecto |
| `max_iter` / `n_estimators` | Tope de pasos | Con early stopping, casi no importa si es alto | Si es bajo, el tope corta antes que la paciencia | Generoso: 500–1000 |
| `learning_rate` | Tamaño de cada paso | Necesita **menos** pasos (pero curva más brusca) | Necesita **más** pasos (curva más suave, a menudo algo mejor) | 0.1 (boosting) |

**Valores por defecto en scikit-learn:**

| Modelo | ¿Activado por defecto? | Paciencia por defecto |
|---|---|---|
| `HistGradientBoostingClassifier/Regressor` | `"auto"`: solo si hay > 10,000 filas | 10 |
| `GradientBoostingClassifier/Regressor` | No (`n_iter_no_change=None`) | — |
| `SGDClassifier/Regressor` | No | 5 |
| `MLPClassifier/Regressor` | No | 10 |

### 🧠 Regla mnemotécnica
> **Pon un tope alto y deja que la validación decida.** La paciencia es el tiempo que esperas antes de creer que ya no mejora.

### 🔮 Ejercicio 8 — El `"auto"` traicionero
Tienes 8,000 filas y entrenas `HistGradientBoostingClassifier(max_iter=1000)` **sin tocar nada más**. ¿Cuántos árboles crees que usará?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Los 1,000.** Con `early_stopping="auto"` (el valor por defecto), solo se activa si hay **más de 10,000** filas. Con 8,000 no hay vigilancia y el modelo usará todos los árboles que le permitas, con riesgo de sobreajuste. Si quieres early stopping, **actívalo explícitamente**: `early_stopping=True`.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo para en las primeras épocas y rinde poco | Paciencia muy baja con una curva nerviosa | Subir `n_iter_no_change`; bajar `learning_rate` |
| Resultados excelentes en prueba que luego no se repiten | Usaste el **conjunto de prueba** como validación para early stopping: **fuga de datos** | Vigilar con una validación sacada del entrenamiento; la prueba, solo al final |
| Early stopping "no hace nada": usa todos los pasos | Desactivado (`"auto"` con pocos datos) o `max_iter` demasiado bajo | `early_stopping=True` y `max_iter` generoso |
| Con muy pocos datos, cada reentrenamiento para en un punto distinto | La validación es tan pequeña que es puro ruido | Subir `validation_fraction`, usar validación cruzada, o no usar early stopping |
| Comparas modelos y el "mejor" usó la prueba para decidir cuándo parar | Misma fuga, más sutil | Toda decisión (incluida cuándo parar) se toma **sin** mirar la prueba |

El segundo error merece atención: **elegir cuándo parar es una decisión**, igual que elegir `alpha` o `max_depth`. Si la tomas mirando la prueba, tu resultado final deja de ser honesto (Lección 2.6).

### ✋ Ejercicio 9 — Detecta el error
Un compañero escribe:

```python
red = MLPClassifier(max_iter=1000)
mejor, mejor_epoca = 0, 0
for epoca in range(1000):
    red.partial_fit(X_train, y_train, classes=[0, 1])
    if red.score(X_test, y_test) > mejor:
        mejor, mejor_epoca = red.score(X_test, y_test), epoca
print("Exactitud final:", mejor)
```

¿Qué está mal? ¿Cómo lo arreglarías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Usa **`X_test`** para decidir en qué época quedarse, y luego reporta la exactitud **en ese mismo `X_test`**. Eligió el mejor de 1,000 intentos mirando el examen: el número reportado es **optimista** (fuga de datos).

Arreglo: separa una **validación** del entrenamiento (`train_test_split(X_train, y_train, ...)`), vigila con ella, y evalúa en `X_test` **una sola vez** al final. O, más simple: `MLPClassifier(early_stopping=True)`, que hace la separación por ti.
</details>

---
## 8. 📋 Ficha de la técnica

| | Early stopping |
|---|---|
| **Tipo** | Técnica de regularización para modelos que se entrenan **por pasos** |
| **Intuición** | Mirar las galletas por la ventanita y sacarlas cuando dejan de mejorar |
| **Cómo funciona** | Vigila una validación tras cada paso y para tras `n_iter_no_change` pasos sin mejora |
| **Perillas clave** | `n_iter_no_change` (paciencia), `validation_fraction`, `tol` |
| **✅ Fortalezas** | Evita el sobreajuste · Ahorra tiempo de cómputo · Te libra de ajustar el número de iteraciones |
| **❌ Debilidades** | Gasta datos en la validación · Con curvas nerviosas o pocos datos, puede parar en mal momento |
| **Úsalo con…** | Gradient boosting, redes neuronales, modelos SGD con muchos datos |
| **No aplica a…** | Modelos que no se entrenan por pasos: árboles individuales, KNN, Naive Bayes |

## 9. 🧠 Resumen en 5 puntos
1. En modelos que aprenden **por pasos**, la pérdida de validación suele bajar y luego **subir**: los pasos extra memorizan ruido.
2. **Early stopping** vigila una validación y **para** cuando deja de mejorar. Es una forma de **regularización**.
3. La **paciencia** (`n_iter_no_change`) evita parar por un bache; demasiado baja hace que pare antes de tiempo.
4. Receta práctica: **tope alto (`max_iter`) + early stopping**, en vez de ajustar el número de pasos a mano.
5. La validación de early stopping sale **del entrenamiento**: usar la prueba para decidir cuándo parar es fuga de datos.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** La pérdida de entrenamiento baja sin parar y la de validación empezó a subir en la época 40. Estás en la época 200. ¿Qué pasa?
- a) Subajuste · b) Sobreajuste desde la época 40 · c) Todo va bien · d) La tasa de aprendizaje es 0

**2.** Con `n_iter_no_change=5`, el modelo…
- a) Hace exactamente 5 iteraciones
- b) Para tras 5 iteraciones seguidas sin mejorar en validación
- c) Mejora 5 veces y para
- d) Ignora las 5 primeras iteraciones

**3.** ¿En cuál de estos modelos **no** tiene sentido early stopping?
- a) `HistGradientBoostingClassifier` · b) `MLPClassifier` · c) `KNeighborsClassifier` · d) `SGDClassifier`

**4.** Tu `SGDClassifier` con early stopping para siempre en la época 3–4 y rinde poco. ¿Qué pruebas primero?
- a) Subir la paciencia · b) Bajar la paciencia · c) Usar la prueba como validación · d) Bajar `max_iter`

**5.** Tienes 5,000 filas y usas `HistGradientBoostingRegressor(max_iter=2000)` sin más parámetros. ¿Cuántos árboles usará?
- a) Unos pocos, gracias a early stopping · b) 2,000 · c) 100 · d) Depende de la paciencia

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La brecha creciente es la firma del sobreajuste. Las últimas 160 épocas solo memorizaron ruido.
2. **b)** Esa es la paciencia.
3. **c)** KNN no se entrena por pasos: solo guarda los datos (Lección 5.3). No hay "cuándo parar".
4. **a)** Paciencia baja + curva nerviosa = parada prematura (Sección 4).
5. **b)** Con 5,000 filas, `"auto"` **no** activa early stopping (umbral: 10,000). Usará los 2,000.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Si early stopping ya decide el número de árboles, ¿por qué seguiría importando el `learning_rate`? (Pista: piensa en dar 50 pasos largos o 500 pasos cortos para bajar una montaña, Lección 3.3.)

**B. Caso de análisis 💼** Una startup entrena una red neuronal que tarda **6 horas** por entrenamiento. El jefe propone: *"Entrenemos siempre 200 épocas, es lo que hicimos la primera vez y salió bien"*. ¿Qué le propondrías y qué ganaría la empresa?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque el `learning_rate` cambia **la forma** de la curva. Con pasos largos, el modelo llega rápido al fondo pero lo hace de forma brusca y puede "pasarse"; con pasos cortos, necesita más iteraciones pero suele encontrar un fondo un poco más bajo. Early stopping decide **cuándo** parar, pero el `learning_rate` decide **cómo** se recorre el camino. Lo comprobarás en la práctica integradora.

**B.** Activar early stopping con un tope de 200 épocas. Ganancias: (1) si el mejor punto está en la época 60, ahorra más de 4 horas por entrenamiento; (2) protege contra el sobreajuste cuando cambian los datos (200 épocas pudieron funcionar la primera vez por suerte); (3) un número menos que ajustar a mano. Riesgo a vigilar: con una paciencia muy baja, podría parar antes de tiempo.
</details>

### 10.3 🧩 Práctica integradora: el tope, la paciencia y la tasa de aprendizaje

Una empresa de logística quiere predecir **el tiempo de entrega** a partir de 10 variables (solo 5 influyen de verdad). Usarás `HistGradientBoostingRegressor` y early stopping para encontrar una buena configuración **sin tocar la prueba**.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos**

In [ ]:
X_e, y_e = make_friedman1(n_samples=3_000, n_features=10, noise=1.0, random_state=0)
X_e_tr, X_e_te, y_e_tr, y_e_te = train_test_split(X_e, y_e, test_size=0.25, random_state=0)
print(f"Entrenamiento: {X_e_tr.shape} · Prueba: {X_e_te.shape}")

**Paso 2 🔮 — Sin vigilancia y con pasos largos.** Entrenamos con `learning_rate=0.5` y 1,000 árboles, sin early stopping. ¿Qué esperas ver en R² de entrenamiento vs. validación cruzada?

✍️ *Mi apuesta:*

In [ ]:
sin_vigilar = HistGradientBoostingRegressor(learning_rate=0.5, max_iter=1000, early_stopping=False, random_state=0)
sin_vigilar.fit(X_e_tr, y_e_tr)
cv = cross_val_score(sin_vigilar, X_e_tr, y_e_tr, cv=5).mean()
print(f"R² entrenamiento: {sin_vigilar.score(X_e_tr, y_e_tr):.3f} · R² validación cruzada: {cv:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento ≈ **1.000** (memorizó) y validación cruzada ≈ **0.89**. Hay brecha: sobreajuste. Con pasos tan largos y 1,000 árboles, el modelo se pasó de largo.
</details>

**Paso 3 ▶️ — Activa early stopping.** Mismo modelo, pero ahora vigilado.

In [ ]:
vigilado = HistGradientBoostingRegressor(learning_rate=0.5, max_iter=1000, early_stopping=True, random_state=0)
cv = cross_val_score(vigilado, X_e_tr, y_e_tr, cv=5).mean()
vigilado.fit(X_e_tr, y_e_tr)
print(f"Árboles usados: {vigilado.n_iter_} · R² validación cruzada: {cv:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Usa solo **19** árboles de 1,000 y obtiene la **misma** validación cruzada (≈ 0.89) en una fracción del tiempo. Con pasos tan largos, el modelo llega a su mejor punto muy rápido y los árboles extra casi no estorbaban aquí. Early stopping ahorró tiempo, pero no puede arreglar unos pasos **demasiado bruscos**. Para eso está la siguiente perilla.
</details>

**Paso 4 ✏️ — Ahora, la tasa de aprendizaje.** Con early stopping siempre activado, prueba varias `learning_rate`. Anota cuántos árboles usa cada una y su R² en validación cruzada.

In [ ]:
TASAS = [0.5, 0.2, 0.1, 0.05]     # ✏️ prueba también 0.02

for lr in TASAS:
    m = HistGradientBoostingRegressor(learning_rate=lr, max_iter=2000, early_stopping=True, random_state=0)
    cv = cross_val_score(m, X_e_tr, y_e_tr, cv=5).mean()
    m.fit(X_e_tr, y_e_tr)
    print(f"learning_rate={lr:<5} → árboles: {m.n_iter_:>4} · R² validación cruzada: {cv:.4f}")

✋ ¿Qué relación ves entre la tasa y el número de árboles? ¿Qué tasa elegirías?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Cuanto **más pequeña** la tasa, **más árboles** necesita (19 con 0.5 → 67 → 132 → 177 con 0.05): pasos cortos, más pasos.
- El R² mejora al bajar la tasa de 0.5 (≈ 0.888) a 0.2 (≈ 0.921) y 0.1 (≈ 0.928), y luego casi no cambia (0.05 → ≈ 0.929).

Una buena elección es **0.1**: empata en la práctica con 0.05 (la diferencia está en el cuarto decimal) y usa menos árboles. Si dos opciones empatan, elige la **más barata**. Esta es la receta estándar del boosting que verás en la Lección 7.4: **tasa pequeña + early stopping decide los árboles**.
</details>

**Paso 5 ▶️ — Examen final, una sola vez.**

In [ ]:
TASA_ELEGIDA = 0.1     # ✏️ pon la tasa que elegiste

final = HistGradientBoostingRegressor(learning_rate=TASA_ELEGIDA, max_iter=2000,
                                      early_stopping=True, random_state=0).fit(X_e_tr, y_e_tr)
print(f"Modelo final       → R² prueba: {final.score(X_e_te, y_e_te):.3f} ({final.n_iter_} árboles)")
print(f"Modelo sin vigilar → R² prueba: {sin_vigilar.score(X_e_te, y_e_te):.3f} (1000 árboles)")

<details><summary><b>👀 Ver respuesta</b></summary>

El modelo final logra ≈ **0.93** en prueba, contra ≈ **0.91** del modelo sin vigilar con pasos largos. Lo conseguiste sin tocar la prueba hasta el final y sin adivinar el número de árboles: early stopping lo eligió por ti.
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, ejemplo *Early stopping in Gradient Boosting* y guía de `HistGradientBoostingClassifier` (sección *Early stopping*).
- 📺 **Video:** Andrew Ng, *Deep Learning Specialization*, curso 2, "Other regularization methods" (early stopping), en inglés con subtítulos.
- 🔜 **En el Módulo 9 (Lecciones 9.6 y 9.8):** early stopping con paciencia en un bucle de PyTorch, guardando los mejores pesos. En Keras existe el *callback* `EarlyStopping(restore_best_weights=True)`.

---
⬅️ **Anterior:** [7.1 Regularización: castigar la complejidad](7.1-regularizacion.ipynb) · ➡️ **Siguiente:** [7.3 Estrategias de búsqueda: cuadrícula, aleatoria, bayesiana y *successive halving*](7.3-estrategias-de-busqueda.ipynb)